Purpose: Load ICA-cleaned epochs from the MNE-BIDS pipeline and save as
compact - npz files ready for the ECOC decoding pipeline.
Pipeline position: Runs AFTER icalabel-vep derivatives are produced.
Do NOT substitute raw EDF - filtering, ICA, and autoreject are already done.
Preprocessing applied upstream (MNE-BIDS pipeline):
• Bandpass: 1.0 - 40.0 Hz
• Epoching: -0.199 to 0.797 s, baseline corrected
• Downsampled: 1024 Hz → 256 Hz
• ICA (Picard): eye/muscle components removed
• Autoreject: bad epochs dropped; globally bad channels physically removed from file
Why some subjects have fewer than 64 channels:
Autoreject detected some electrodes as bad for the entire session (poor scalp
contact, dried gel, etc.) and deleted them completely from the fif file.
Different subjects lost different channels (e.g. sub-1001 lost P1, P2, PO4).
Without fixing this, subjects would have inconsistent feature dimensions,
breaking the ECOC decoder which assumes a fixed (trials × 64 x 256) input.
This script restores all 64 channels via spherical spline interpolation:
Missing channels are first added as zero-filled placeholders (they do not exist
in the file at all - MNE cannot mark non-existent channels as bad), then
interpolated from their spatial neighbours using spherical splines.
This mirrors the supervisor's MATLAB pipeline which always saved _interpol.set
files with exactly 64 channels before decoding.
Output per subject x pathway:
x (trials × 64ch × 256tp, float32), y (0-indexed labels [0-3)), times, ch_names, age

In [5]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
from pathlib import Path
import mne
import numpy as np
import logging

In [6]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────

# Logging: message-only format for clean output
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

# ── Paths ─────────────────────────────────────────────────────────────────────
# DERIV_DIR: output of the MNE-BIDS icalabel pipeline.
# Each subject folder contains: sub-XXXX_task-vep_proc-clean_epo.fif
DERIV_DIR = Path("/Volumes/cmvm/scs/groups/HELIOS-BD/Part B/hbd_vep/derivatives/icalabel-vep")
SAVE_ROOT = Path("/Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/Final_VEP/DATA_NPZ")

# ── Subjects (49 participants with age data) ──────────────────────────────────
SUBJECTS = [
    '1001', '1002', '1004', '1005', '1007', '1008', '1010', '1011', '1014', '1016',
    '1017', '1018', '1020', '1021', '1023', '1024', '1026', '1028', '1034', '1038',
    '1039', '1041', '1042', '1044', '1046', '1052', '2002', '2006', '2009', '2017',
    '2020', '2023', '2026', '2028', '2029', '2037', '3001', '3005', '3006', '3007',
    '3008', '3011', '3014', '3016', '3027', '3030', '3034', '3039', '3041'
]

# ── Age data ──────────────────────────────────────────────────────────────────
# Ages are not in participants.tsv (anonymised at source), so stored here.
# Sorted by subject ID for readability.
AGE_DATA = {
    '1001': 25.0, '1002': 43.0, '1004': 33.0, '1005': 50.0, '1007': 31.0,
    '1008': 25.0, '1010': 26.0, '1011': 37.0, '1014': 23.0, '1016': 36.0,
    '1017': 46.0, '1018': 25.0, '1020': 29.0, '1021': 28.0, '1023': 46.0,
    '1024': 24.0, '1026': 25.0, '1028': 62.0, '1034': 60.0, '1038': 45.0,
    '1039': 63.0, '1041': 49.0, '1042': 39.0, '1044': 62.0, '1046': 61.0,
    '1052': 53.0, '2002': 62.0, '2006': 25.0, '2009': 49.0, '2017': 31.0,
    '2020': 43.0, '2023': 49.0, '2026': 47.0, '2028': 29.0, '2029': 50.0,
    '2037': 45.0, '3001': 54.0, '3005': 39.0, '3006': 33.0, '3007': 41.0,
    '3008': 32.0, '3011': 31.0, '3014': 34.0, '3016': 44.0, '3027': 72.0,
    '3030': 54.0, '3034': 48.0, '3039': 34.0, '3041': 35.0,
}

# ── Pathway definitions ───────────────────────────────────────────────────────
# 'codes': string event IDs as stored in the clean .fif (confirmed from HTML report).
# Order = contrast level 1 (lowest) → 4 (highest).
# This order produces 0-indexed y labels [0, 1, 2, 3] for the ECOC decoder.
PATHWAYS = {
    "Luminance": {"codes": ["Lum/1", "Lum/2", "Lum/3", "Lum/4"], "out": "Luminance_NPZ"},
    "L-M":       {"codes": ["LM/1",  "LM/2",  "LM/3",  "LM/4"],  "out": "L_M_NPZ"},
    "S-cone":    {"codes": ["S/1",   "S/2",   "S/3",   "S/4"],    "out": "S_cone_NPZ"},
}

# ── Sanity checks ─────────────────────────────────────────────────────────────
missing_age = [s for s in SUBJECTS if s not in AGE_DATA]
if missing_age:
    logger.warning(f"Subjects missing from AGE_DATA: {missing_age}")
else:
    logger.info(f"Age data verified for all {len(SUBJECTS)} subjects ✓")

logger.info(f"Derivatives : {DERIV_DIR}")
logger.info(f"Output root : {SAVE_ROOT}")

Age data verified for all 49 subjects ✓
Derivatives : /Volumes/cmvm/scs/groups/HELIOS-BD/Part B/hbd_vep/derivatives/icalabel-vep
Output root : /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/Final_VEP/DATA_NPZ


In [8]:
# ── Cell 3: Processing function ───────────────────────────────────────────────

# Load the BioSemi 64-channel montage once — reused for every subject.
# _BIOSEMI64_NAMES is the canonical ordered list of 64 electrode names.
_BIOSEMI64_MONTAGE = mne.channels.make_standard_montage('biosemi64')
_BIOSEMI64_NAMES   = _BIOSEMI64_MONTAGE.ch_names  # ordered list, length 64


def process_subject_pathway(sub_id, pathway_name, config):
    """
    Load cleaned epochs for one subject + one pathway and save as .npz.

    Parameters
    ----------
    sub_id       : str   e.g. '1001'
    pathway_name : str   e.g. 'Luminance'
    config       : dict  entry from PATHWAYS — keys: 'codes', 'out'

    Saved arrays
    ------------
    X        : float32  (trials, 64, 256)  ICA-cleaned, interpolated EEG
    y        : int32    (trials,)          0-indexed contrast labels [0,1,2,3]
    times    : float64  (256,)             epoch time axis in seconds
    ch_names : str      (64,)              BioSemi64 channel names
    age      : float32  scalar             participant age in years
    """
    sub_str = f"sub-{sub_id}"

    # ── 1. Locate the clean epochs file ──────────────────────────────────────
    # Output of the MNE-BIDS icalabel pipeline: bandpass filtered (1–40 Hz),
    # ICA-cleaned (Picard), autoreject applied, downsampled to 256 Hz.
    # Do NOT use raw .edf — it bypasses all cleaning steps.
    epo_path = (
        DERIV_DIR / sub_str / "eeg" / f"{sub_str}_task-vep_proc-clean_epo.fif"
    )
    if not epo_path.exists():
        logger.warning(f"MISSING: {epo_path}")
        return

    # ── 2. Load clean epochs ──────────────────────────────────────────────────
    epochs = mne.read_epochs(epo_path, preload=True, verbose=False)

    # ── 3. Keep only EEG channels ─────────────────────────────────────────────
    # Drop any non-EEG channels (EOG, misc, status) that survived the pipeline.
    # After this step a subject may have 61–64 EEG channels depending on how
    # many were physically removed as globally bad during autoreject.
    epochs.pick_types(eeg=True, verbose=False)

    # ── 4. Restore physically missing channels → always exactly 64 ───────────
    #
    # PROBLEM: The pipeline physically deleted some channels from the .fif file
    # (e.g. P1, P2, PO4 for sub-1001). They do not exist anywhere in the file.
    # MNE's interpolate_bads() can only fix channels that already exist in the
    # epochs object — it raises a ValueError if we try to mark a non-existent
    # channel as bad. So we must add them first as zero placeholders.
    #
    # SOLUTION (three steps):
    #   A. Add each missing channel as a flat-zero EpochsArray
    #   B. Re-apply the montage so new channels get 3-D scalp positions
    #   C. Mark as bad → interpolate from neighbours → clear bads list

    # Apply montage so existing channels already have positions
    epochs.set_montage(_BIOSEMI64_MONTAGE, on_missing='ignore', verbose=False)

    # Identify channels present in BioSemi64 standard but absent from this subject
    missing_ch = [ch for ch in _BIOSEMI64_NAMES if ch not in epochs.ch_names]

    if missing_ch:
        # Step A — create zero-filled placeholder epochs for missing channels
        info_missing = mne.create_info(
            ch_names = missing_ch,
            sfreq    = epochs.info['sfreq'],
            ch_types = 'eeg'
        )
        zero_data   = np.zeros((len(epochs), len(missing_ch), len(epochs.times)))
        placeholder = mne.EpochsArray(
            zero_data, info_missing, tmin=epochs.tmin, verbose=False
        )

        # Merge placeholders into the main epochs object
        epochs.add_channels([placeholder], force_update_info=True)

        # Step B — re-apply montage so newly added channels get 3-D positions
        # (spherical spline interpolation requires positions for all channels)
        epochs.set_montage(_BIOSEMI64_MONTAGE, on_missing='ignore', verbose=False)

        # Step C — mark as bad and interpolate via spherical splines.
        # Channels now exist in the object so MNE accepts them as bads.
        # reset_bads=True clears the bads list after interpolation is done.
        epochs.info['bads'] = missing_ch
        epochs.interpolate_bads(reset_bads=True, verbose=False)

        logger.info(
            f"{sub_str}: interpolated {len(missing_ch)} channel(s): "
            f"{sorted(missing_ch)}"
        )

    # Hard assertion: every subject must now have exactly 64 channels
    assert len(epochs.ch_names) == 64, (
        f"{sub_str}: expected 64 channels after interpolation, "
        f"got {len(epochs.ch_names)}"
    )

    # ── 5. Select trials for this pathway using string event IDs ─────────────
    # Event IDs in the .fif are strings: 'Lum/1', 'LM/2', 'S/4', etc.
    # (confirmed from sub-1001 MNE-BIDS pipeline HTML report)
    available     = set(epochs.event_id.keys())
    requested     = set(config['codes'])
    found         = requested & available
    missing_codes = requested - available

    if not found:
        logger.warning(
            f"{sub_str} | {pathway_name}: no matching events found. "
            f"Available: {sorted(available)}"
        )
        return

    if missing_codes:
        logger.warning(
            f"{sub_str} | {pathway_name}: missing contrast levels "
            f"{sorted(missing_codes)}"
        )

    epochs = epochs[sorted(found)]

    if len(epochs) == 0:
        logger.warning(f"{sub_str} | {pathway_name}: 0 trials after event selection")
        return

    # ── 6. Build 0-indexed integer labels ────────────────────────────────────
    # Remap string event IDs → [0, 1, 2, 3] so the ECOC decoder receives
    # contiguous class labels regardless of original trigger codes.
    # Example: 'Lum/1'→0, 'Lum/2'→1, 'Lum/3'→2, 'Lum/4'→3
    code_to_idx = {code: i for i, code in enumerate(config['codes'])}
    id_to_str   = {v: k for k, v in epochs.event_id.items()}  # int → string
    y = np.array(
        [code_to_idx[id_to_str[ev]] for ev in epochs.events[:, 2]],
        dtype=np.int32
    )

    # ── 7. Save as compressed NPZ ─────────────────────────────────────────────
    out_dir = SAVE_ROOT / config['out']
    out_dir.mkdir(parents=True, exist_ok=True)

    np.savez_compressed(
        out_dir / f"{sub_str}_{pathway_name}_data.npz",
        X        = epochs.get_data().astype(np.float32),  # (trials,64,256) float32 ~50% smaller
        y        = y,                                      # (trials,) int32, values in [0,1,2,3]
        times    = epochs.times,                           # (256,) float64, -0.199–0.797 s
        ch_names = np.array(epochs.ch_names),              # (64,) BioSemi64 channel names
        age      = np.float32(AGE_DATA[sub_id]),           # scalar — used in aging analysis
    )

    logger.info(f"SUCCESS: {sub_str} | {pathway_name} | {len(epochs)} epochs")
    del epochs  # free RAM before next iteration

In [9]:
# ── Cell 4: Run — all subjects × all pathways ─────────────────────────────────
for sub in SUBJECTS:
    for pathway_name, config in PATHWAYS.items():
        process_subject_pathway(sub, pathway_name, config)

Created an SSP operator (subspace dimension = 1)
1 projection items activated


sub-1001: interpolated 3 channel(s): ['P1', 'P2', 'PO4']
SUCCESS: sub-1001 | Luminance | 238 epochs


Created an SSP operator (subspace dimension = 1)
1 projection items activated


sub-1001: interpolated 3 channel(s): ['P1', 'P2', 'PO4']
SUCCESS: sub-1001 | L-M | 239 epochs


Created an SSP operator (subspace dimension = 1)
1 projection items activated


...

sub-3041: interpolated 3 channel(s): ['P2', 'PO3', 'Pz']
SUCCESS: sub-3041 | Luminance | 240 epochs


Created an SSP operator (subspace dimension = 1)
1 projection items activated


sub-3041: interpolated 3 channel(s): ['P2', 'PO3', 'Pz']
SUCCESS: sub-3041 | L-M | 239 epochs


Created an SSP operator (subspace dimension = 1)
1 projection items activated


sub-3041: interpolated 3 channel(s): ['P2', 'PO3', 'Pz']
SUCCESS: sub-3041 | S-cone | 240 epochs


In [10]:
# ── Cell 5: Verification ──────────────────────────────────────────────────────
# Spot-check three subjects: youngest (1001, age 25), middle (1028, age 62),
# and oldest (3027, age 72). All must show identical X shape (N, 64, 256),
# confirming that interpolation produced a consistent channel layout.
CHECK_SUBJECTS = ['1001', '1028', '3027']

print(f"{'Pathway':<15} | {'Subject':<10} | {'X Shape':<20} | "
      f"{'y labels':<15} | {'Age':<6} | {'Time range'}")
print("-" * 90)

for pathway_name, config in PATHWAYS.items():
    folder = SAVE_ROOT / config['out']
    files  = sorted(folder.glob("*.npz")) if folder.exists() else []

    if not files:
        print(f"{pathway_name:<15} | ❌ No files found")
        continue

    for f in files:
        sid = f.stem.split('_')[0].replace('sub-', '')
        if sid not in CHECK_SUBJECTS:
            continue
        # allow_pickle=False: safe — only plain numpy arrays are stored
        with np.load(f, allow_pickle=False) as d:
            x_shape  = d['X'].shape
            y_unique = np.unique(d['y'])
            age      = float(d['age'])
            t_min    = float(d['times'][0])
            t_max    = float(d['times'][-1])

        print(f"{pathway_name:<15} | sub-{sid:<8} | {str(x_shape):<20} | "
              f"{str(y_unique):<15} | {age:<6.1f} | {t_min:.3f}–{t_max:.3f} s")
    print()

# Total file count — guard against missing folders
total = sum(
    len(list((SAVE_ROOT / c['out']).glob('*.npz')))
    for c in PATHWAYS.values()
    if (SAVE_ROOT / c['out']).exists()
)
expected = len(SUBJECTS) * len(PATHWAYS)
status   = '✅' if total == expected else '⚠️'
print(f"Total files : {total} / {expected} {status}  "
      f"({len(SUBJECTS)} subjects × {len(PATHWAYS)} pathways)")
print()
print("Expected X shape  : (≈240, 64, 256)   — trials × channels × timepoints")
print("Expected y labels : [0 1 2 3]          — 0-indexed contrast levels")
print("Expected time     : -0.199 – 0.797 s   — epoch window at 256 Hz")

Pathway         | Subject    | X Shape              | y labels        | Age    | Time range
------------------------------------------------------------------------------------------
Luminance       | sub-1001     | (238, 64, 256)       | [0 1 2 3]       | 25.0   | -0.199–0.797 s
Luminance       | sub-1028     | (229, 64, 256)       | [0 1 2 3]       | 62.0   | -0.199–0.797 s
Luminance       | sub-3027     | (230, 64, 256)       | [0 1 2 3]       | 72.0   | -0.199–0.797 s

L-M             | sub-1001     | (239, 64, 256)       | [0 1 2 3]       | 25.0   | -0.199–0.797 s
L-M             | sub-1028     | (238, 64, 256)       | [0 1 2 3]       | 62.0   | -0.199–0.797 s
L-M             | sub-3027     | (223, 64, 256)       | [0 1 2 3]       | 72.0   | -0.199–0.797 s

S-cone          | sub-1001     | (239, 64, 256)       | [0 1 2 3]       | 25.0   | -0.199–0.797 s
S-cone          | sub-1028     | (233, 64, 256)       | [0 1 2 3]       | 62.0   | -0.199–0.797 s
S-cone          | sub-3027     